# Check input testuali per embeddings

Questo notebook valida i file JSONL gia' preparati per lo scenario di invoice matching semantico.

Obiettivi:
- controllare conteggi, schema e qualita' minima di `embedding_text`;
- verificare che il testo resti focalizzato su segnali semantici e non includa campi strutturati vietati;
- controllare che le query abbiano `evaluation_split` nei metadata con valori ammessi;
- produrre un riepilogo finale dei problemi trovati.

Non vengono chiamati modelli embedding, vector database o API FastAPI.

## 1. Import e percorsi

In [ ]:
import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


pd.set_option("display.max_colwidth", 220)

current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
DATA_DIR = PROJECT_ROOT / "data" / "processed"

CORPUS_PATH = DATA_DIR / "embedding_corpus.jsonl"
QUERIES_PATH = DATA_DIR / "embedding_queries.jsonl"

CORPUS_PATH, QUERIES_PATH

## 2. Caricamento JSONL

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON non valido in {path} alla riga {line_number}: {exc}") from exc
    return records


corpus_records = read_jsonl(CORPUS_PATH)
query_records = read_jsonl(QUERIES_PATH)

datasets = {
    "corpus": corpus_records,
    "queries": query_records,
}

pd.DataFrame(
    [
        {"dataset": name, "records": len(records), "path": str(path)}
        for name, records, path in [
            ("corpus", corpus_records, CORPUS_PATH),
            ("queries", query_records, QUERIES_PATH),
        ]
    ]
)

## 3. Controllo schema

Ogni record deve contenere `record_id`, `source_image`, `embedding_text` e `metadata`.

In [ ]:
REQUIRED_FIELDS = {"record_id", "source_image", "embedding_text", "metadata"}


def check_schema(records: list[dict[str, Any]], dataset_name: str) -> pd.DataFrame:
    issues = []
    for row_number, record in enumerate(records, start=1):
        keys = set(record)
        missing_fields = sorted(REQUIRED_FIELDS - keys)
        extra_fields = sorted(keys - REQUIRED_FIELDS)

        if missing_fields or extra_fields:
            issues.append(
                {
                    "dataset": dataset_name,
                    "row_number": row_number,
                    "record_id": record.get("record_id"),
                    "issue": "field_set",
                    "details": f"missing={missing_fields}; extra={extra_fields}",
                }
            )

        type_checks = {
            "record_id": lambda value: value is not None,
            "source_image": lambda value: isinstance(value, str) and bool(value.strip()),
            "embedding_text": lambda value: isinstance(value, str),
            "metadata": lambda value: isinstance(value, dict),
        }
        for field, is_valid in type_checks.items():
            if field in record and not is_valid(record.get(field)):
                issues.append(
                    {
                        "dataset": dataset_name,
                        "row_number": row_number,
                        "record_id": record.get("record_id"),
                        "issue": "invalid_type_or_value",
                        "details": field,
                    }
                )

    return pd.DataFrame(issues)


schema_issues = pd.concat(
    [check_schema(records, name) for name, records in datasets.items()],
    ignore_index=True,
)

schema_issues.head(20)

## 4. Testi vuoti o troppo corti

In [ ]:
MIN_TEXT_CHARS = 50


def find_short_texts(records: list[dict[str, Any]], dataset_name: str) -> pd.DataFrame:
    rows = []
    for row_number, record in enumerate(records, start=1):
        text = record.get("embedding_text", "")
        text = text if isinstance(text, str) else ""
        stripped_text = text.strip()
        if not stripped_text or len(stripped_text) < MIN_TEXT_CHARS:
            rows.append(
                {
                    "dataset": dataset_name,
                    "row_number": row_number,
                    "record_id": record.get("record_id"),
                    "source_image": record.get("source_image"),
                    "text_length_chars": len(stripped_text),
                    "embedding_text": stripped_text,
                }
            )
    return pd.DataFrame(rows)


short_text_issues = pd.concat(
    [find_short_texts(records, name) for name, records in datasets.items()],
    ignore_index=True,
)

short_text_issues.head(20)

## 5. Statistiche lunghezza `embedding_text`

In [ ]:
def text_length_stats(records: list[dict[str, Any]], dataset_name: str) -> dict[str, Any]:
    lengths = np.array([len(str(record.get("embedding_text", "")).strip()) for record in records])
    word_counts = np.array([len(str(record.get("embedding_text", "")).split()) for record in records])

    percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
    result = {
        "dataset": dataset_name,
        "records": len(records),
        "min_chars": int(lengths.min()) if len(lengths) else None,
        "max_chars": int(lengths.max()) if len(lengths) else None,
        "mean_chars": round(float(lengths.mean()), 2) if len(lengths) else None,
        "median_chars": round(float(np.median(lengths)), 2) if len(lengths) else None,
        "mean_words": round(float(word_counts.mean()), 2) if len(word_counts) else None,
        "median_words": round(float(np.median(word_counts)), 2) if len(word_counts) else None,
    }
    for percentile in percentiles:
        result[f"p{percentile}_chars"] = round(float(np.percentile(lengths, percentile)), 2) if len(lengths) else None
    return result


length_stats = pd.DataFrame([text_length_stats(records, name) for name, records in datasets.items()])
length_stats

## 6. Numero medio di descrizioni prodotto

In [ ]:
PRODUCT_DESCRIPTION_LINE_PATTERN = re.compile(r"^\s*-\s+\S+", flags=re.MULTILINE)


def count_product_descriptions(text: Any) -> int:
    if not isinstance(text, str):
        return 0
    return len(PRODUCT_DESCRIPTION_LINE_PATTERN.findall(text))


product_description_stats = []
for dataset_name, records in datasets.items():
    counts = np.array([count_product_descriptions(record.get("embedding_text")) for record in records])
    product_description_stats.append(
        {
            "dataset": dataset_name,
            "min_product_descriptions": int(counts.min()) if len(counts) else None,
            "max_product_descriptions": int(counts.max()) if len(counts) else None,
            "mean_product_descriptions": round(float(counts.mean()), 2) if len(counts) else None,
            "median_product_descriptions": round(float(np.median(counts)), 2) if len(counts) else None,
            "records_without_product_descriptions": int((counts == 0).sum()) if len(counts) else None,
        }
    )

product_description_stats = pd.DataFrame(product_description_stats)
product_description_stats

## 7. Esempi casuali

In [ ]:
RANDOM_STATE = 42
SAMPLE_SIZE = 5


def sample_embedding_texts(records: list[dict[str, Any]], dataset_name: str, sample_size: int = SAMPLE_SIZE) -> pd.DataFrame:
    frame = pd.DataFrame(
        [
            {
                "dataset": dataset_name,
                "record_id": record.get("record_id"),
                "source_image": record.get("source_image"),
                "embedding_text": record.get("embedding_text"),
            }
            for record in records
        ]
    )
    return frame.sample(n=min(sample_size, len(frame)), random_state=RANDOM_STATE).reset_index(drop=True)


random_examples = pd.concat(
    [sample_embedding_texts(records, name) for name, records in datasets.items()],
    ignore_index=True,
)

random_examples

## 8. Etichette o sezioni vietate nel testo

Questi controlli cercano label strutturate che devono restare fuori da `embedding_text`: invoice number, invoice date, quantity, unit price, total price, subtotal, total, discount e VAT.

In [ ]:
FORBIDDEN_LABEL_PATTERNS = {
    "invoice number": re.compile(r"\binvoice\s*(number|no\.?|#)\b", flags=re.IGNORECASE),
    "invoice date": re.compile(r"\binvoice\s*date\b", flags=re.IGNORECASE),
    "quantity": re.compile(r"\b(quantity|qty)\b", flags=re.IGNORECASE),
    "unit price": re.compile(r"\bunit\s*price\b", flags=re.IGNORECASE),
    "total price": re.compile(r"\btotal\s*price\b", flags=re.IGNORECASE),
    "subtotal": re.compile(r"\b(subtotal|sub\s*total)\b", flags=re.IGNORECASE),
    "total": re.compile(r"\btotal\b", flags=re.IGNORECASE),
    "discount": re.compile(r"\bdiscount\b", flags=re.IGNORECASE),
    "VAT": re.compile(r"\bVAT\b", flags=re.IGNORECASE),
}


def find_forbidden_text_labels(records: list[dict[str, Any]], dataset_name: str) -> pd.DataFrame:
    rows = []
    for row_number, record in enumerate(records, start=1):
        text = record.get("embedding_text", "")
        text = text if isinstance(text, str) else ""
        for label, pattern in FORBIDDEN_LABEL_PATTERNS.items():
            match = pattern.search(text)
            if match:
                start = max(match.start() - 60, 0)
                end = min(match.end() + 60, len(text))
                rows.append(
                    {
                        "dataset": dataset_name,
                        "row_number": row_number,
                        "record_id": record.get("record_id"),
                        "source_image": record.get("source_image"),
                        "forbidden_label": label,
                        "matched_text": match.group(0),
                        "context": text[start:end].replace("\n", " | "),
                    }
                )
    return pd.DataFrame(rows)


forbidden_text_issues = pd.concat(
    [find_forbidden_text_labels(records, name) for name, records in datasets.items()],
    ignore_index=True,
)

forbidden_text_issues.head(50)

## 9. Controllo `evaluation_split` nelle queries

In [ ]:
ALLOWED_QUERY_SPLITS = {"validation", "test"}


def check_query_evaluation_split(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for row_number, record in enumerate(records, start=1):
        metadata = record.get("metadata")
        split = metadata.get("evaluation_split") if isinstance(metadata, dict) else None
        if split not in ALLOWED_QUERY_SPLITS:
            rows.append(
                {
                    "dataset": "queries",
                    "row_number": row_number,
                    "record_id": record.get("record_id"),
                    "source_image": record.get("source_image"),
                    "evaluation_split": split,
                    "issue": "missing_or_invalid_evaluation_split",
                }
            )
    return pd.DataFrame(rows)


query_split_issues = check_query_evaluation_split(query_records)

query_split_counts = pd.Series(
    [record.get("metadata", {}).get("evaluation_split") for record in query_records if isinstance(record.get("metadata"), dict)]
).value_counts(dropna=False).rename_axis("evaluation_split").reset_index(name="records")

display(query_split_counts)
query_split_issues.head(20)

## 10. Riepilogo finale

In [ ]:
summary_rows = [
    {"check": "schema", "issues": len(schema_issues)},
    {"check": "embedding_text_empty_or_too_short", "issues": len(short_text_issues)},
    {"check": "forbidden_text_labels", "issues": len(forbidden_text_issues)},
    {"check": "query_evaluation_split", "issues": len(query_split_issues)},
]

summary = pd.DataFrame(summary_rows)
total_issues = int(summary["issues"].sum())

display(summary)

if total_issues == 0:
    print("OK: nessun problema trovato nei controlli principali.")
else:
    print(f"ATTENZIONE: trovati {total_issues} problemi nei controlli principali.")
    print("Ispezionare le tabelle schema_issues, short_text_issues, forbidden_text_issues e query_split_issues.")